# Import and setup

In [1]:
import os
import re
import numpy as np
import pandas as pd
import polars as pl
from towbintools.foundation.image_handling import read_tiff_file
from towbintools.foundation.image_quality import normalized_variance_measure
import matplotlib.pyplot as plt
from tifffile import imwrite
from towbintools.foundation.file_handling import add_dir_to_experiment_filemap
from skimage.measure import shannon_entropy
from scipy.optimize import curve_fit
import random

def pick_random_crops(stack_path, planes_per_stack = 1, crops_per_plane=1, n_crops=1, tile_size=None, channels_to_keep=[1], score_threshold=1.5):
    stack = read_tiff_file(stack_path, channels_to_keep=[1])

    scores = [normalized_variance_measure(plane) for plane in stack]

    fit = np.polyfit(np.arange(len(scores)), scores, 3)

    plt.plot(np.arange(len(scores)), scores)
    plt.plot(np.arange(len(scores)), np.polyval(fit, np.arange(len(scores))))
    plt.show()

    scores = np.polyval(fit, np.arange(len(scores)))

    best_plane = np.argmax(scores)
    planes_with_data = stack[best_plane-5:best_plane+6]

    try:
        random_planes = np.random.choice(
            np.arange(planes_with_data.shape[0]),
            size=planes_per_stack,
            replace=False
        )
    except ValueError as e:
        print("Not enough planes to choose from:", e)
        return

    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        crops = []

        if tile_size is None:
            crops.append(random_plane)
        else:
            for _ in range(n_crops):
                x = np.random.randint(0, random_plane.shape[0] - tile_size)
                y = np.random.randint(0, random_plane.shape[1] - tile_size)
                crop = random_plane[x:x+tile_size, y:y+tile_size]
                crops.append(crop)


            # adjusted_crops = [crop - 100 for crop in crops]
            # adjusted_crops = [np.clip(crop - 100, 0, None) for crop in crops]
            adjusted_crops = crops
            crops = np.array(crops)
            scores = [normalized_variance_measure(crop) for crop in adjusted_crops]

            # remove crops with too low scores
            scores = np.array(scores)
            
            valid_indices = np.where(scores > score_threshold)[0]
            scores = scores[valid_indices]
            crops = crops[valid_indices]

            if crops.shape[0] < crops_per_plane:
                print("Not enough valid crops to choose from.")
                continue

            if crops.shape[0] < crops_per_plane:
                print("Not enough valid crops to choose from.")
                continue

            chosen_indexes = np.random.choice(
                np.arange(crops.shape[0]),
                size=crops_per_plane,
                replace=False
            )

            crops = crops[chosen_indexes]

    return crops

def pick_random_planes(stack_path, planes_per_stack = 1, channels_to_keep=[0, 1], autofocus_channel = 0):
    stack = read_tiff_file(stack_path, channels_to_keep=channels_to_keep)
    autofocus_stack = stack[:, autofocus_channel]

    scores = [normalized_variance_measure(plane) for plane in autofocus_stack]
    # cut the scores +- 9 frames around the max
    scores = np.array(scores)
    max_index = np.argmax(scores)
    start = max(0, max_index - 8)
    end = min(len(scores), max_index + 9)
    scores = scores[start:end]
    fit = np.polyfit(np.arange(len(scores)), scores, 4)

    scores = np.polyval(fit, np.arange(len(scores)))

    best_plane = np.argmax(scores)

    # for our images, interesting planes are typically around -10 to +5 of the best plane
    start_plane = max(0, best_plane - 7)
    end_plane = min(stack.shape[0], best_plane + 4)
    planes_with_data = stack[start_plane:end_plane]

    try:
        random_planes = np.random.choice(
            np.arange(planes_with_data.shape[0]),
            size=planes_per_stack,
            replace=False
        )
    except ValueError as e:
        print("Not enough planes to choose from:", e)
        return

    planes = []
    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        planes.append(random_plane)

    return planes

def save(images, output_dir, hash_size = 5):
    random_hex_hash = os.urandom(hash_size).hex()
    if images is None or len(images) == 0:
        print("No images to save.")
        return
    for image in images:
        imwrite(os.path.join(output_dir, f"crop_{random_hex_hash}.tiff"), image, compression="zlib")

In [ ]:
experiment_dir = "/mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter"
stacks_dir = os.path.join(experiment_dir, "raw_stacks")

output_dir = "/mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/datasets/yap1_GFP_60x_confocal/"
output_dir = os.path.join(output_dir, "raw")
os.makedirs(output_dir, exist_ok=True)

stacks_to_pick = 1ch1chiijijijijjij00
planes_per_stack=2
crops_per_plane=1
n_crops=1
tile_size=None
channels_to_keep=[0, 1]
score_threshold=1.5

final_channel_to_keep = [0]
final_output_dir = os.path.join(output_dir, "gfp", "images")
os.makedirs(final_output_dir, exist_ok=True)

# Fully annotated

In [3]:
from towbintools.foundation.file_handling import read_filemap

filemap_path = os.path.join(experiment_dir, "analysis", "report", "analysis_filemap_annotated.parquet")
try:
    filemap = read_filemap(filemap_path)
    filemap = add_dir_to_experiment_filemap(filemap, stacks_dir, subdir_name="raw_zstack")
except FileNotFoundError as e:
    print("File not found:", e)
# convert filemap to pandas 
filemap = filemap.to_pandas()
# keep only rows that are not ignored
filemap = filemap[~filemap['Ignore']]
filemap['Time'] = pd.to_numeric(filemap['Time'], errors='coerce')
filemap = filemap[filemap['raw_zstack'] != '']

ecdysis = ['HatchTime', 'M1', 'M2', 'M3', 'M4']
for e in ecdysis:
    filemap[e] = pd.to_numeric(filemap[e], errors='coerce')

stage_proportions = {
    'egg': 0.0,
    'L1': 0.5,
    'L2': 0.5,
    'L3': 0.0,
    'L4': 0.0,
    'adult': 0.0
}
for i, (stage, proportion) in enumerate(stage_proportions.items()):
    if stage == 'egg':
        stage_filemap = filemap[filemap['Time'] < filemap[ecdysis[0]]]
    elif stage == 'adult':
        stage_filemap = filemap[filemap['Time'] > filemap[ecdysis[-1]]]
    else:
        stage_filemap = filemap[
            (filemap['Time'] > filemap[ecdysis[i-1]]) &
            (filemap['Time'] < filemap[ecdysis[i]])
        ]

    n_stacks = stacks_to_pick*proportion
    print(f"Selecting {n_stacks} stacks for stage '{stage}'")
    if n_stacks < 1:
        continue
    
    selected_filemap = stage_filemap.sample(n=int(n_stacks))

    for _, row in selected_filemap.iterrows():
        stack_path = row['raw_zstack']
        print("Processing stack:", stack_path)
        crops = pick_random_planes(
            stack_path,
            planes_per_stack=planes_per_stack,
            channels_to_keep=channels_to_keep
        )
        save(crops, output_dir)
        save([crop[final_channel_to_keep[0]] for crop in crops], final_output_dir)

Selecting 0.0 stacks for stage 'egg'
Selecting 100.0 stacks for stage 'L1'
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter/raw_stacks/Time00012_Point0036_ChannelGFP,mCherry_Seq2056.ome.tiff
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter/raw_stacks/Time00027_Point0047_ChannelGFP,mCherry_Seq4558.ome.tiff
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter/raw_stacks/Time00045_Point0014_ChannelGFP,mCherry_Seq7468.ome.tiff
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter/raw_stacks/Time00012_Point0001_ChannelGFP,mCherry_Seq1986.ome.tiff
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIVA_60x_707_708_col10_reporter/raw_stacks/Time00063_Point0099_ChannelGFP,mCherry_Seq10614.ome.tiff
Processing stack: /mnt/towbin.data/shared/spsalmon/20260807_134551_682_ZIV

# With minimal annotations

In [4]:
filemap_path = "/mnt/towbin.data/shared/spsalmon/20251014_150718_923_ZIVA_60x_443_additional_stardist_training_data/part1/analysis/report/analysis_filemap_annotated.csv"
directory_to_add = "/mnt/towbin.data/shared/spsalmon/20251014_150718_923_ZIVA_60x_443_additional_stardist_training_data/part1/raw_zstack"
filemap = pd.read_csv(filemap_path)
filemap = add_dir_to_experiment_filemap(filemap, directory_to_add, subdir_name="raw_zstack")

output_dir = "/mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/db2_part1"
output_dir = os.path.join(output_dir, "raw")
os.makedirs(output_dir, exist_ok=True)

print(filemap.columns)
# keep only rows that are not ignored
filemap = filemap[~filemap['Ignore']]

# for each point, keep only rows where Time > HatchTime if HatchTime is not NaN
# convert Time to numeric
filemap['Time'] = pd.to_numeric(filemap['Time'], errors='coerce')
# convert HatchTime to numeric
filemap['HatchTime'] = pd.to_numeric(filemap['HatchTime'], errors='coerce')

filemap = filemap[(filemap['HatchTime'].isna()) | (filemap['Time'] > filemap['HatchTime'])]
filemap.head(77)

interesting_stacks = filemap['raw_zstack'].tolist()
# remove empty strings
interesting_stacks = [s for s in interesting_stacks if s != ""]

print(f"Number of interesting stacks: {len(interesting_stacks)}")

number_of_stacks = 100
planes_per_stack = 2
np.random.shuffle(interesting_stacks)
picked_stacks = interesting_stacks[:number_of_stacks]

ValueError: Joining multiple DataFrames only supported for joining on index

In [ ]:
for stack in picked_stacks:
    test_stack = read_tiff_file(stack, channels_to_keep=[1])

    scores = []
    for i in range(test_stack.shape[0]):
        slice = test_stack[i]
        score = normalized_variance_measure(slice)
        scores.append(score)

    best_plane = np.argmax(scores)
    planes_with_data = test_stack[best_plane-2:]

    try:
        random_planes = np.random.choice(
            np.arange(planes_with_data.shape[0]),
            size=planes_per_stack,
            replace=False
        )
    except ValueError as e:
        print("Not enough planes to choose from:", e)
        continue

    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        tile_size = 512
        # take N random crops of size tile_size x tile_size
        num_crops = 50
        crops = []
        for _ in range(num_crops):
            x = np.random.randint(0, random_plane.shape[0] - tile_size)
            y = np.random.randint(0, random_plane.shape[1] - tile_size)
            crop = random_plane[x:x+tile_size, y:y+tile_size]
            crops.append(crop)

        crops = np.array(crops)
        scores = []
        for crop in crops:
            score = normalized_variance_measure(crop)
            scores.append(score)

        # remove crops with too low scores
        scores = np.array(scores)
        print(np.median(scores), np.mean(scores), np.std(scores))
        valid_indices = np.where(scores > 1.)[0]
        scores = scores[valid_indices]
        crops = crops[valid_indices]

        try:
            # randomly pick a crop among the top 10 crops
            top_indices = np.argsort(scores)[-10:]
            chosen_index = np.random.choice(top_indices)
            best_crop = crops[chosen_index]
            random_hex_hash = os.urandom(4).hex()
            imwrite(os.path.join(output_dir, f"crop_{random_hex_hash}.tiff"), best_crop, compression="zlib")
            
        except Exception as e:
            print("Not enough valid crops to choose from:", e)

0.683307543650477 0.6787534786952686 0.024041337089829348
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.7318693940261753 0.7385616824514527 0.07809755790746695
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.8915528622479951 0.8920120036059883 0.1336353795836347
0.7350227262915634 0.7840962590398989 0.09858730243599681
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6611799269269065 0.6605471510818752 0.024064445363017026
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6692420508866717 0.6682869180173757 0.018191918417536614
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6564209239733159 0.6564870488729142 0.017251085070149174
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.7686582808540425 0.7827514069604387 0.11964736711153531
0.7214120832

# No annotations

In [ ]:
import os
import re
import numpy as np
from towbintools.foundation.image_handling import read_tiff_file
from towbintools.foundation.image_quality import normalized_variance_measure
import matplotlib.pyplot as plt
from tifffile import imwrite
experiment_dir = "/mnt/towbin.data/shared/kstojanovski/20251212_Ziva_40x_TS_RNAi_test_wBT392-450-462-467_25C_20251212_152743_931"
stacks_dir = os.path.join(experiment_dir, "raw_zstack")

output_dir = "/mnt/towbin.data/shared/kstojanovski/stardist_database/40x_full_nuclei/"
output_dir = os.path.join(output_dir, "raw")
os.makedirs(output_dir, exist_ok=True)

all_zstack_files = [
    os.path.join(stacks_dir, f)
    for f in os.listdir(stacks_dir)
    if f.endswith(".tiff")
]

number_of_stacks = 100
planes_per_stack = 2
np.random.shuffle(all_zstack_files)
picked_stacks = all_zstack_files[:number_of_stacks]

In [ ]:
for stack in picked_stacks:
    test_stack = read_tiff_file(stack, channels_to_keep=[1])

    scores = []
    for i in range(test_stack.shape[0]):
        slice = test_stack[i]
        score = normalized_variance_measure(slice)
        scores.append(score)

    best_plane = np.argmax(scores)
    print(f'Best plane index: {best_plane} with score {scores[best_plane]}')
    planes_with_data = test_stack[best_plane-3:best_plane+15]

    random_planes = np.random.choice(
        np.arange(planes_with_data.shape[0]),
        size=planes_per_stack,
        replace=False
    )

    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        tile_size = 512
        # take N random crops of size tile_size x tile_size
        num_crops = 50
        crops = []
        for _ in range(num_crops):
            x = np.random.randint(0, random_plane.shape[0] - tile_size)
            y = np.random.randint(0, random_plane.shape[1] - tile_size)
            crop = random_plane[x:x+tile_size, y:y+tile_size]
            crops.append(crop)

        crops = np.array(crops)
        scores = []
        for crop in crops:
            score = normalized_variance_measure(crop)
            scores.append(score)

        # remove crops with too low scores
        scores = np.array(scores)
        print(np.median(scores), np.mean(scores), np.std(scores))
        valid_indices = np.where(scores > 1.)[0]
        scores = scores[valid_indices]
        crops = crops[valid_indices]

        try:
            # randomly pick a crop among the top 10 crops
            top_indices = np.argsort(scores)[-10:]
            chosen_index = np.random.choice(top_indices)
            best_crop = crops[chosen_index]
            random_hex_hash = os.urandom(4).hex()
            imwrite(os.path.join(output_dir, f"crop_{random_hex_hash}.tiff"), best_crop, compression="zlib")
            
        except Exception as e:
            print("Not enough valid crops to choose from:", e)



Best plane index: 11 with score 2078.17965473059
1764.206822010267 1715.6624442998127 430.81211042267165
1094.0066567260146 1024.0947707757 259.7203324865381
Best plane index: 7 with score 264.70924002626583
87.04075871485401 71.15626068331204 52.459922352468546
34.02626036222247 43.9386543033503 32.97979075992421
Best plane index: 8 with score 0.8676525199540308
0.7228162025882807 0.812746186109851 0.1587087056227318
0.699384822411161 0.7701257147516948 0.13213473937216946
Best plane index: 5 with score 0.9865619948013583
0.7083468175184369 0.8792774905986159 0.30107282458744355
0.7780799508940013 1.1002579090910347 0.42195451363160663
Best plane index: 12 with score 240.77027717593543
79.79050923686819 60.58016934219576 44.190881819529906
237.89273139080916 227.02934718179117 147.70211163101075
Best plane index: 8 with score 2036.0527380151673
725.715357615335 676.8611330921855 612.6459258615846
1451.036382258208 1577.8247667440185 1335.8752119340736
Best plane index: 1 with score 32

ValueError: 'a' cannot be empty unless no samples are taken